# RNA-seq Survival and Mutation Integration Template

用于 TCGA 或其它带临床结局队列的单基因/签名生存分析、Cox 回归、突变 MAF 概览、表达-突变整合。通常接在 TCGA/GEO 模板或已有表达矩阵之后。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
EXPR_FILE <- "./data/expression_matrix.csv"  # genes x samples; counts/TPM/VST all可用，但生存图建议log表达
CLINICAL_FILE <- "./data/clinical.csv"
MAF_FILE <- NULL                         # e.g. "./data/tcga.maf"; NULL to skip mutation analysis
GENE_COLUMN <- NULL
SAMPLE_COLUMN <- "sample"
PATIENT_ID_EXPR_CHARS <- 12              # TCGA barcode patient id length
PATIENT_ID_CLINICAL <- "submitter_id"
OS_TIME_COLUMN <- "days_to_death"
OS_FOLLOWUP_COLUMN <- "days_to_last_follow_up"
OS_EVENT_COLUMN <- "vital_status"        # "Dead" vs "Alive"

TARGET_GENES <- c("CD274", "CXCL9", "CXCL10", "GZMB")
SIGNATURES <- list(
  IFNG_signature = c("IFNG", "CXCL9", "CXCL10", "STAT1", "IDO1"),
  Cytotoxicity = c("GZMA", "GZMB", "PRF1", "NKG7", "GNLY")
)
GROUP_METHOD <- "median"                 # "median" or "tertile"
COVARIATES <- NULL                        # e.g. c("age_at_index", "ajcc_pathologic_stage")
OUTDIR <- "RNAseq_Survival_Mutation_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse", "survival", "survminer", "pheatmap", "broom"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install("maftools")

suppressPackageStartupMessages({
  library(tidyverse)
  library(survival)
  library(survminer)
  library(pheatmap)
  library(broom)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")


## 3. Load Expression and Clinical Data

In [ ]:
expr_raw <- read.csv(EXPR_FILE, check.names = FALSE)
if (!is.null(GENE_COLUMN) && GENE_COLUMN %in% colnames(expr_raw)) {
  genes <- expr_raw[[GENE_COLUMN]]
  expr <- as.matrix(expr_raw[, setdiff(colnames(expr_raw), GENE_COLUMN), drop = FALSE])
  rownames(expr) <- genes
} else if (!is.numeric(expr_raw[[1]])) {
  genes <- expr_raw[[1]]
  expr <- as.matrix(expr_raw[, -1, drop = FALSE])
  rownames(expr) <- genes
} else {
  expr <- as.matrix(expr_raw)
}
mode(expr) <- "numeric"
expr <- expr[!duplicated(rownames(expr)) & rownames(expr) != "", , drop = FALSE]
expr_log <- log2(expr + 1)

clinical <- read.csv(CLINICAL_FILE, check.names = FALSE)
clinical$patient_id <- clinical[[PATIENT_ID_CLINICAL]]
clinical$OS_time <- suppressWarnings(as.numeric(clinical[[OS_TIME_COLUMN]]))
followup <- suppressWarnings(as.numeric(clinical[[OS_FOLLOWUP_COLUMN]]))
clinical$OS_time[is.na(clinical$OS_time)] <- followup[is.na(clinical$OS_time)]
clinical$OS_event <- ifelse(clinical[[OS_EVENT_COLUMN]] == "Dead", 1, 0)
clinical <- clinical[!is.na(clinical$OS_time) & clinical$OS_time > 0 & !is.na(clinical$OS_event), ]

sample_df <- data.frame(sample = colnames(expr_log), patient_id = substr(colnames(expr_log), 1, PATIENT_ID_EXPR_CHARS))
cat("Expression:", nrow(expr_log), "genes x", ncol(expr_log), "samples\n")
cat("Clinical patients:", nrow(clinical), "\n")


## 4. Build Gene and Signature Scores

In [ ]:
score_df <- sample_df
for (gene in intersect(TARGET_GENES, rownames(expr_log))) {
  score_df[[gene]] <- as.numeric(expr_log[gene, score_df$sample])
}

row_upper <- toupper(rownames(expr_log))
upper_to_real <- setNames(rownames(expr_log), row_upper)
for (sig_name in names(SIGNATURES)) {
  matched <- unique(upper_to_real[intersect(toupper(SIGNATURES[[sig_name]]), names(upper_to_real))])
  if (length(matched) >= 2) {
    score_df[[sig_name]] <- colMeans(expr_log[matched, score_df$sample, drop = FALSE], na.rm = TRUE)
    cat(sig_name, ":", length(matched), "genes matched\n")
  }
}

surv_df <- score_df %>% group_by(patient_id) %>% summarise(across(where(is.numeric), mean, na.rm = TRUE), .groups = "drop") %>%
  inner_join(clinical, by = "patient_id")
write.csv(surv_df, file.path(OUTDIR, "survival_score_table.csv"), row.names = FALSE)
cat("Survival analysis samples:", nrow(surv_df), "\n")


## 5. Kaplan-Meier Analysis

In [ ]:
make_group <- function(x, method = GROUP_METHOD) {
  if (method == "tertile") {
    qs <- quantile(x, probs = c(1/3, 2/3), na.rm = TRUE)
    out <- ifelse(x <= qs[1], "Low", ifelse(x >= qs[2], "High", NA))
  } else {
    out <- ifelse(x >= median(x, na.rm = TRUE), "High", "Low")
  }
  factor(out, levels = c("Low", "High"))
}

score_cols <- setdiff(colnames(score_df), c("sample", "patient_id"))
km_summary <- list()
for (feature in score_cols) {
  df <- surv_df %>% select(patient_id, OS_time, OS_event, all_of(feature)) %>% filter(!is.na(.data[[feature]]))
  df$group <- make_group(df[[feature]])
  df <- df[!is.na(df$group), ]
  if (length(unique(df$group)) < 2 || nrow(df) < 10) next
  fit <- survfit(Surv(OS_time, OS_event) ~ group, data = df)
  p <- ggsurvplot(fit, data = df, pval = TRUE, risk.table = TRUE,
                  title = paste(feature, "overall survival"), legend.title = feature)
  pdf(file.path(OUTDIR, paste0("KM_", feature, ".pdf")), width = 7, height = 7)
  print(p)
  dev.off()
  cox <- coxph(Surv(OS_time, OS_event) ~ group, data = df)
  km_summary[[feature]] <- broom::tidy(cox, exponentiate = TRUE, conf.int = TRUE)
}
if (length(km_summary) > 0) write.csv(bind_rows(km_summary, .id = "feature"), file.path(OUTDIR, "KM_cox_summary.csv"), row.names = FALSE)


## 6. Multivariable Cox Analysis

In [ ]:
if (!is.null(COVARIATES)) {
  cox_multi <- list()
  for (feature in score_cols) {
    cov_keep <- COVARIATES[COVARIATES %in% colnames(surv_df)]
    fml <- as.formula(paste("Surv(OS_time, OS_event) ~", paste(c(feature, cov_keep), collapse = " + ")))
    df <- surv_df[, unique(c("OS_time", "OS_event", feature, cov_keep)), drop = FALSE]
    df <- df[complete.cases(df), ]
    if (nrow(df) < 20) next
    fit <- coxph(fml, data = df)
    cox_multi[[feature]] <- broom::tidy(fit, exponentiate = TRUE, conf.int = TRUE)
  }
  if (length(cox_multi) > 0) write.csv(bind_rows(cox_multi, .id = "feature"), file.path(OUTDIR, "Multivariable_cox_summary.csv"), row.names = FALSE)
}


## 7. Mutation Analysis with maftools

In [ ]:
if (!is.null(MAF_FILE)) {
  if (!requireNamespace("maftools", quietly = TRUE)) stop("Install maftools before mutation analysis.")
  maf <- maftools::read.maf(MAF_FILE)
  pdf(file.path(OUTDIR, "MAF_oncoplot.pdf"), width = 10, height = 8)
  maftools::oncoplot(maf = maf, top = 30)
  dev.off()
  maf_summary <- maftools::getSampleSummary(maf)
  write.csv(maf_summary, file.path(OUTDIR, "MAF_sample_summary.csv"), row.names = FALSE)

  mutated_genes <- intersect(TARGET_GENES, maftools::getGeneSummary(maf)$Hugo_Symbol)
  for (gene in mutated_genes) {
    pdf(file.path(OUTDIR, paste0("Lollipop_", gene, ".pdf")), width = 8, height = 5)
    maftools::lollipopPlot(maf = maf, gene = gene)
    dev.off()
  }
}
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
